# 🚀 JAVDL
### Google Colab Runner

> **One-click setup → Telegram bot → Javtiful downloader**

This notebook runs the `javdl` branch in a fresh Google Colab runtime.

### ✨ What it does
- 📦 Installs the required dependencies
- 🔐 Loads Telegram credentials securely from Colab Secrets
- ⚡ Configures the current multi-client Telegram uploader
- 🧪 Runs a pre-flight check
- 🤖 Starts the JAVDL Telegram bot

### 🔐 Required Secrets
`TELEGRAM_API_ID` · `TELEGRAM_API_HASH` · `TELEGRAM_BOT_TOKEN`

**Run the cells from top to bottom.**


## 🧰 1 · Prepare JAVDL

Clone the latest `javdl` branch into the Colab runtime.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/Telebotfaroff/javtiful-scraper.git"
BRANCH = "javdl"
WORKDIR = Path("/content/javdl")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORKDIR)],
    check=True,
)
os.chdir(WORKDIR)
print(f"JAVDL {BRANCH} ready: {Path.cwd()}")


## 📦 2 · Install dependencies

Installs FFmpeg, Python packages and Chromium for Playwright fallback extraction.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!python -m pip install -q -r requirements.txt
!python -m playwright install --with-deps chromium


## 🔐 3 · Configure Telegram

Open **Colab → Secrets** and add:

| Secret | Required |
|---|---|
| `TELEGRAM_API_ID` | ✅ |
| `TELEGRAM_API_HASH` | ✅ |
| `TELEGRAM_BOT_TOKEN` | ✅ |

Your secret values are loaded at runtime and are not written to the repository.


In [ ]:
from google.colab import userdata

required = ["TELEGRAM_API_ID", "TELEGRAM_API_HASH", "TELEGRAM_BOT_TOKEN"]
secrets = {}

for name in required:
    try:
        secrets[name] = userdata.get(name)
    except Exception as exc:
        raise RuntimeError(f"Unable to read Colab Secret {name!r}: {exc}") from exc

missing = [name for name, value in secrets.items() if not value]
if missing:
    raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))

os.environ.update({
    **{name: str(value) for name, value in secrets.items()},
    "TELEGRAM_SESSION": "javdl_colab",
    "TELEGRAM_UPLOAD_SESSION": "javdl_uploads",
    "TELEGRAM_UPLOAD_CLIENTS": "2",
    "TELEGRAM_MAX_CONCURRENT_TRANSMISSIONS": "8",
})

print("Telegram credentials loaded. Upload clients: 2")


## 📋 4 · Enable live logs

Useful diagnostics for extraction, downloads, FFmpeg, thumbnails and Telegram uploads.


In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    stream=sys.stdout,
    force=True,
)

for logger_name in ("app", "app.extractor", "app.downloader", "app.uploaders", "app.jobs", "app.bot"):
    logging.getLogger(logger_name).setLevel(logging.INFO)

print("Application logging enabled.")


## 🧪 5 · Run pre-flight check

Compile the application and verify that the Telegram bot factory imports correctly before starting anything.


In [ ]:
!python -m compileall -q app main.py scripts/colab.py

from app.bot.telegram import create_app

print("JAVDL imports and syntax check passed.")


## 🤖 6 · Start JAVDL

### Bot flow
`Javtiful URL` → `Preview` → `Download` → `Telegram upload`

Channel/studio URLs can also be sent to the bot for batch downloading.

> **Important:** Colab already has an asyncio event loop. Do not use `app.run()` here.

▶️ Run the cell below and keep it running while you use the bot.


In [ ]:
from pyrogram import idle
from app.bot.telegram import create_app

app = create_app()

print("🚀 Starting JAVDL Telegram bot…")
await app.start()
print("")
print("╔══════════════════════════════════════╗")
print("║        ✅ JAVDL BOT IS ONLINE        ║")
print("╚══════════════════════════════════════╝")
print("")
print("📨 Send a Javtiful URL to the bot.")
print("🛑 Stop with the Colab interrupt button.")

try:
    await idle()
finally:
    if app.is_connected:
        await app.stop()
    print("🛑 JAVDL bot stopped.")
